In [1]:
pip install tensorflow==2.15

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 1.16.0
    Uninstalling wrapt-1.16.0:
      Successfully uninstalled wrapt-1.16.0
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.4.0
    Uninstalling ml-dtypes-0.4.0:
      Successfully uninstalled ml-dtypes-0.4.0
  Attempting uninstall: keras
    Found existing installation: keras 3.4.1
    Uninstalling keras-3.4.1:
      Successfully uninstalled keras-3.4.1
  Attempting uninstall: tensorboard
    Found existing installation

In [2]:
# libraries
import os
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator
import time

#Visualizers
from yellowbrick.classifier import ClassificationReport
from yellowbrick.classifier import ClassPredictionError
from yellowbrick.classifier import ConfusionMatrix
from yellowbrick.classifier import ROCAUC
from yellowbrick.classifier import PrecisionRecallCurve
import matplotlib.pyplot as plt

#Metrics
#Metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import hamming_loss
from sklearn.metrics import log_loss
from sklearn.metrics import zero_one_loss
from sklearn.metrics import matthews_corrcoef
from sklearn.metrics import precision_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score
from sklearn.metrics import classification_report



#Classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

#Neural Network
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense,LSTM
from tensorflow.keras.layers import Conv1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Lambda, Layer, ReLU
from keras.models import load_model
import tensorflow as tf
from tensorflow.keras.losses import sparse_categorical_crossentropy
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D,Concatenate

import warnings
warnings.filterwarnings('ignore')

In [3]:
from keras import optimizers

In [4]:
import tensorflow
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, Dense, Input, Reshape, Lambda, Layer, Flatten, BatchNormalization,AveragePooling2D
from tensorflow.keras import backend as K
import matplotlib.pyplot as plt
import numpy as np
from keras.datasets import mnist
from imblearn.over_sampling import SMOTE

from keras import initializers

from keras.utils import to_categorical


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
df=pd.read_excel("/content/drive/MyDrive/PaperExtracciónDental/dataset_extraccion_up_low_bimax.xlsx")

In [7]:
df.drop(columns=['Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48',"Extraccion / no extraccion inf","Extraccion / no extraccion sup", "Extraccion / no extraccion","Paciente","Extraccion tejidos blandos",'discrepancia total inferior', 'discrepancia total superior',"Clase real"],inplace=True)

In [8]:
from sklearn.preprocessing import LabelEncoder

In [9]:
LabelEncoder_1=LabelEncoder()
df["Genero"]=LabelEncoder_1.fit_transform(df["Genero"])
df["Etnia"]=LabelEncoder_1.fit_transform(df["Etnia"])
df["Clasificación  esqueletica"]=LabelEncoder_1.fit_transform(df["Clasificación  esqueletica"])
df["Relación molar"]=LabelEncoder_1.fit_transform(df["Relación molar"])
df["Relacion canina"]=LabelEncoder_1.fit_transform(df["Relacion canina"])
df["Relación premolar"]=LabelEncoder_1.fit_transform(df["Relación premolar"])


In [10]:
df = df[df['Tipo de extraccióin global'] != 'Extracción inf.']


In [11]:
df['Tipo de extraccióin global'] = df['Tipo de extraccióin global'].map({'Extracción bima': 1, 'No extracción': 0,"Extracción sup.":1})

#Capsule-Net

In [12]:
from keras.layers import GlobalAveragePooling2D,ReLU
num_clases=2
vec=16
class DigitCapsuleLayer(Layer):
    # creating a layer class in keras
    def __init__(self, **kwargs):
        super(DigitCapsuleLayer, self).__init__(**kwargs)
        self.kernel_initializer = initializers.get('glorot_uniform')

    def build(self, input_shape):
        # initialize weight matrix for each capsule in lower layer
        self.W = self.add_weight(shape = [1,1,num_clases, vec, 256], initializer = self.kernel_initializer, name = 'weights') #:c
        self.built = True

    def call(self, inputs):
        #print(inputs.shape)
        u = tf.reshape(inputs, (-1, 1, 256)) # u.shape: (None, 1152, 8) #:c

        u = tf.expand_dims(u, axis=-2) # u.shape: (None, 1152, 1, 8)
        #print(u.shape)
        u = tf.expand_dims(u, axis=-1) # u.shape: (None, 1152, 1, 8, 1)
        u_hat = tf.matmul(self.W, u) # u_hat.shape: (None, 1152, 10, 16, 1)
        u_hat = tf.squeeze(u_hat, [4]) # u_hat.shape: (None, 1152, 10, 16)
        b = tf.zeros(shape = [K.shape(inputs)[0],2, num_clases, 1])

# routing algorithm with updating coupling coefficient c, using scalar product b/w input capsule and output capsule
        for i in range(3-1):
            c = tf.nn.softmax(b, axis=-2) # c.shape: (None, 1152, 10, 1)
            s = tf.reduce_sum(tf.multiply(c, u_hat), axis=1, keepdims=True) # s.shape: (None, 1, 10, 16)
            v = squash(s) # v.shape: (None, 1, 10, 16)
            agreement = tf.squeeze(tf.matmul(tf.expand_dims(u_hat, axis=-1), tf.expand_dims(v, axis=-1), transpose_a=True), [4]) # agreement.shape: (None, 1152, 10, 1)
            b += agreement

        return v

    def compute_output_shape(self, input_shape):
        return tuple([None, num_clases, vec])

epsilon = 1e-7

def output_layer(inputs):
    return K.sqrt(K.sum(K.square(inputs), -1) + K.epsilon())

def squash(inputs):
    # take norm of input vectors
    squared_norm = tf.keras.backend.sum(tf.keras.backend.square(inputs), axis = -1, keepdims = True)

    # use the formula for non-linear function to return squashed output
    return ((squared_norm/(1+squared_norm))/(K.sqrt(squared_norm+K.epsilon())))*inputs

def safe_norm(v, axis=-1, epsilon=1e-7):
    v_ = tf.reduce_sum(tf.square(v), axis = axis, keepdims=True)
    return tf.sqrt(v_ + epsilon)


In [13]:

# RED NEURONAL
def FullyConnected():


  inputs = Input(shape=(X_train.shape[1],1), name="input_1")


  Layer_1=tf.keras.layers.Conv1D(8,3,activation="selu",padding="same")(inputs)
  #Pool_1=tf.keras.layers.MaxPool1D(2)(Layer_1)


#16

  Layer_1=tf.keras.layers.Conv1D(16,3,activation="selu",padding="same")(Layer_1)
  Pool_1=tf.keras.layers.MaxPool1D(2)(Layer_1)


#32

  Layer_1=tf.keras.layers.Conv1D(32,3,activation="selu",padding="same")(Pool_1)
  #Pool_1=tf.keras.layers.MaxPool1D(2)(Layer_1)
  Pool_1=tf.keras.layers.Dropout(rate=0.5)(Layer_1)

#64

  Layer_1=tf.keras.layers.Conv1D(64,3,activation="selu",padding="same")(Pool_1)
  Pool_1=tf.keras.layers.MaxPool1D(2)(Layer_1)
  Pool_1=tf.keras.layers.Dropout(rate=0.5)(Pool_1)

#128

  Layer_1=tf.keras.layers.Conv1D(128,3,activation="selu",padding="same")(Pool_1)
  Pool_1=tf.keras.layers.MaxPool1D(2)(Layer_1)
  Pool_1=tf.keras.layers.Dropout(rate=0.5)(Pool_1)

#256

  Layer_1=tf.keras.layers.Conv1D(256,3,activation="selu",padding="same")(Pool_1)
  Pool_1=tf.keras.layers.MaxPool1D(2)(Layer_1)
  Pool_1=tf.keras.layers.Dropout(rate=0.5)(Pool_1)

  squashed_output = tf.keras.layers.Lambda(squash)(Pool_1)
  digit_caps = DigitCapsuleLayer()(squashed_output)
  Acont= safe_norm(digit_caps)
  mast = tf.reshape(Acont, (-1,Acont.shape[2],Acont.shape[3]))
  outputs = tf.keras.layers.Lambda(output_layer)(mast)
  model = Model(inputs = inputs, outputs = outputs)

  optimizer=Adam(learning_rate=0.001)
  model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
  return model

In [14]:
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn import decomposition
import numpy as np
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint

# Definición de clases
classes = [0, 1]

# Configuración de la validación cruzada y los datos
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Separación de características y etiquetas
Labels = df['Tipo de extraccióin global']
Features = df.drop(['Tipo de extraccióin global'], axis=1)

# División del dataset en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(Features, Labels, test_size=0.2, stratify=Labels, random_state=42)

scaler = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
# Aplicación de PCA
pca = decomposition.PCA(n_components=0.96, svd_solver='full', tol=1e-4)
pca.fit(X_train)
X_train = pca.transform(X_train)
X_test = pca.transform(X_test)

# Listas para almacenar las métricas de validación cruzada
accuracies = []
f1_scores = []
precisions = []
recalls = []

# Listas para almacenar las métricas del conjunto de prueba
test_accuracies = []
test_f1_scores = []
test_precisions = []
test_recalls = []

# Inicialización del fold
fold_no = 1

# Realizar la validación cruzada
for train_index, val_index in kf.split(X_train):
    print(f'Fold {fold_no}')
    fold_no += 1

    # División de los datos en entrenamiento y validación
    X_train_fold = X_train[train_index]
    y_train_fold = y_train.iloc[train_index]
    smote = SMOTE(random_state=42)
    X_train_fold, y_train_fold = smote.fit_resample(X_train_fold, y_train_fold)
    X_val_fold = X_train[val_index]
    y_val_fold = y_train.iloc[val_index]

    # Expansión de dimensiones para la red neuronal
    X_train_fold = np.expand_dims(X_train_fold, axis=-1)
    X_val_fold = np.expand_dims(X_val_fold, axis=-1)
    X_test_expanded = np.expand_dims(X_test, axis=-1)

    # Conversión de las etiquetas a categóricas
    y_train_fold = to_categorical(y_train_fold.astype(int), 2)
    y_val_fold = to_categorical(y_val_fold.astype(int), 2)
    y_test_categorical = to_categorical(y_test.astype(int), 2)

    # Inicialización del modelo
    model = FullyConnected()

    # Configuración del callback para guardar el mejor modelo basado en la métrica de validación
    checkpoint = ModelCheckpoint(
        'best_model.h5',  # Nombre del archivo donde se guarda el mejor modelo
        monitor='val_loss',  # Métrica a monitorear
        save_best_only=True,  # Guardar solo el mejor modelo
        mode='min',  # Modo de la métrica (minimizar la pérdida)
        verbose=0  # Mostrar mensajes de guardado
    )

    # Entrenamiento del modelo con validación y callback
    model.fit(
        X_train_fold,
        y_train_fold,
        epochs=500,
        batch_size=256,
        validation_data=(X_val_fold, y_val_fold),  # Validación en cada época
        callbacks=[checkpoint],  # Incluir el callback
        verbose=0
    )

    # Cargar el mejor modelo guardado
    model.load_weights('best_model.h5')

    # Hacer predicciones en el conjunto de validación
    y_pred_val = model.predict(X_val_fold)
    y_pred_val_classes = to_categorical(np.argmax(y_pred_val, axis=1), num_classes=2)

    # Calcular las métricas para el conjunto de validación
    accuracy_val = accuracy_score(y_val_fold, y_pred_val_classes) * 100
    f1_val = f1_score(y_val_fold, y_pred_val_classes, average='weighted') * 100
    precision_val = precision_score(y_val_fold, y_pred_val_classes, average='weighted') * 100
    recall_val = recall_score(y_val_fold, y_pred_val_classes, average='weighted') * 100

    # Almacenar las métricas de validación
    accuracies.append(accuracy_val)
    f1_scores.append(f1_val)
    precisions.append(precision_val)
    recalls.append(recall_val)

    # Hacer predicciones en el conjunto de prueba externo
    y_pred_test = model.predict(X_test_expanded)
    y_pred_test_classes = to_categorical(np.argmax(y_pred_test, axis=1), num_classes=2)

    # Calcular las métricas para el conjunto de prueba
    accuracy_test = accuracy_score(y_test_categorical, y_pred_test_classes) * 100
    f1_test = f1_score(y_test_categorical, y_pred_test_classes, average='weighted') * 100
    precision_test = precision_score(y_test_categorical, y_pred_test_classes, average='weighted') * 100
    recall_test = recall_score(y_test_categorical, y_pred_test_classes, average='weighted') * 100

    # Almacenar las métricas del conjunto de prueba
    test_accuracies.append(accuracy_test)
    test_f1_scores.append(f1_test)
    test_precisions.append(precision_test)
    test_recalls.append(recall_test)

# Función para formatear las métricas
def format_metric(mean, std):
    return f'{mean:.2f} ± {std:.2f}'

# Calcular las métricas medias y desviaciones estándar para la validación cruzada
accuracy_mean_std = format_metric(np.mean(accuracies), np.std(accuracies))
f1_mean_std = format_metric(np.mean(f1_scores), np.std(f1_scores))
precision_mean_std = format_metric(np.mean(precisions), np.std(precisions))
recall_mean_std = format_metric(np.mean(recalls), np.std(recalls))

# Calcular las métricas medias y desviaciones estándar para el conjunto de prueba
test_accuracy_mean_std = format_metric(np.mean(test_accuracies), np.std(test_accuracies))
test_f1_mean_std = format_metric(np.mean(test_f1_scores), np.std(test_f1_scores))
test_precision_mean_std = format_metric(np.mean(test_precisions), np.std(test_precisions))
test_recall_mean_std = format_metric(np.mean(test_recalls), np.std(test_recalls))

print(f'\nMétricas de Validación Cruzada:')
print(f'Precisión: {accuracy_mean_std}%')
print(f'Precisión (Precision): {precision_mean_std}%')
print(f'Recall: {recall_mean_std}%')
print(f'F1-score: {f1_mean_std}%')


# Imprimir las métricas del conjunto de prueba
print(f'\nMétricas del Conjunto de Prueba:')
print(f'Precisión: {test_accuracy_mean_std}%')
print(f'Precisión (Precision): {test_precision_mean_std}%')
print(f'Recall: {test_recall_mean_std}%')
print(f'F1-score: {test_f1_mean_std}%')


Fold 1
4/4 [==============================] - 0s 59ms/step
Fold 2
4/4 [==============================] - 0s 3ms/step
Fold 3
4/4 [==============================] - 0s 3ms/step
Fold 4
4/4 [==============================] - 0s 3ms/step
Fold 5
4/4 [==============================] - 0s 3ms/step

Métricas de Validación Cruzada:
Precisión: 92.23 ± 4.18%
Precisión (Precision): 92.25 ± 4.20%
Recall: 92.23 ± 4.18%
F1-score: 92.23 ± 4.19%

Métricas del Conjunto de Prueba:
Precisión: 85.15 ± 1.54%
Precisión (Precision): 85.32 ± 1.72%
Recall: 85.15 ± 1.54%
F1-score: 85.14 ± 1.57%


#CNN

In [15]:
from tensorflow.keras.layers import Input, Dense, Conv1D, MaxPooling1D, Dropout, BatchNormalization, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def FullyConnected():
    # Capa de entrada
    inputs = Input(shape=(X_train.shape[1], 1), name="input_1")

    # Capa convolucional 1
    x = Conv1D(8, 3, activation="selu", padding="same")(inputs)
    x = Conv1D(16, 3, activation="selu", padding="same")(x)
    x = Conv1D(32, 3, activation="selu", padding="same")(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(rate=0.5)(x)

    # Capa convolucional 2
    x = Conv1D(64, 3, activation="selu", padding="same")(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(rate=0.5)(x)

    # Capa convolucional 3
    x = Conv1D(128, 3, activation="selu", padding="same")(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(rate=0.5)(x)

    # Capa convolucional 4
    x = Conv1D(256, 3, activation="selu", padding="same")(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(rate=0.5)(x)

    # Aplanar la salida de las capas convolucionales
    x =tf.keras.layers.GlobalMaxPooling1D()(x)

    # Capas densas
    x = Dense(256, activation="selu")(x)
    x = BatchNormalization()(x)

    x = Dense(128, activation="selu")(x)
    x = BatchNormalization()(x)

    x = Dense(64, activation="selu")(x)
    x = BatchNormalization()(x)

    x = Dense(32, activation="selu")(x)
    x = BatchNormalization()(x)

    x = Dense(16, activation="selu")(x)
    x = BatchNormalization()(x)

    # Capa de salida
    predictions = Dense(2, activation="softmax", name="output_1")(x)

    # Modelo
    model = Model(inputs=inputs, outputs=predictions)
    optimizer = Adam(learning_rate=0.001)

    model.compile(optimizer=optimizer,
                  loss="categorical_crossentropy",
                  metrics=['accuracy'])

    return model


In [16]:
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn import decomposition
import numpy as np
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint

# Definición de clases
classes = [0, 1]

# Configuración de la validación cruzada y los datos
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Separación de características y etiquetas
Labels = df['Tipo de extraccióin global']
Features = df.drop(['Tipo de extraccióin global'], axis=1)

# División del dataset en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(Features, Labels, test_size=0.2, stratify=Labels, random_state=42)

scaler = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
# Aplicación de PCA
pca = decomposition.PCA(n_components=0.96, svd_solver='full', tol=1e-4)
pca.fit(X_train)
X_train = pca.transform(X_train)
X_test = pca.transform(X_test)

# Listas para almacenar las métricas de validación cruzada
accuracies = []
f1_scores = []
precisions = []
recalls = []

# Listas para almacenar las métricas del conjunto de prueba
test_accuracies = []
test_f1_scores = []
test_precisions = []
test_recalls = []

# Inicialización del fold
fold_no = 1

# Realizar la validación cruzada
for train_index, val_index in kf.split(X_train):
    print(f'Fold {fold_no}')
    fold_no += 1

    # División de los datos en entrenamiento y validación
    X_train_fold = X_train[train_index]
    y_train_fold = y_train.iloc[train_index]
    smote = SMOTE(random_state=42)
    X_train_fold, y_train_fold = smote.fit_resample(X_train_fold, y_train_fold)
    X_val_fold = X_train[val_index]
    y_val_fold = y_train.iloc[val_index]

    # Expansión de dimensiones para la red neuronal
    X_train_fold = np.expand_dims(X_train_fold, axis=-1)
    X_val_fold = np.expand_dims(X_val_fold, axis=-1)
    X_test_expanded = np.expand_dims(X_test, axis=-1)

    # Conversión de las etiquetas a categóricas
    y_train_fold = to_categorical(y_train_fold.astype(int), 2)
    y_val_fold = to_categorical(y_val_fold.astype(int), 2)
    y_test_categorical = to_categorical(y_test.astype(int), 2)

    # Inicialización del modelo
    model = FullyConnected()

    # Configuración del callback para guardar el mejor modelo basado en la métrica de validación
    checkpoint = ModelCheckpoint(
        'best_model.h5',  # Nombre del archivo donde se guarda el mejor modelo
        monitor='val_loss',  # Métrica a monitorear
        save_best_only=True,  # Guardar solo el mejor modelo
        mode='min',  # Modo de la métrica (minimizar la pérdida)
        verbose=0  # Mostrar mensajes de guardado
    )

    # Entrenamiento del modelo con validación y callback
    model.fit(
        X_train_fold,
        y_train_fold,
        epochs=500,
        batch_size=256,
        validation_data=(X_val_fold, y_val_fold),  # Validación en cada época
        callbacks=[checkpoint],  # Incluir el callback
        verbose=0
    )

    # Cargar el mejor modelo guardado
    model.load_weights('best_model.h5')

    # Hacer predicciones en el conjunto de validación
    y_pred_val = model.predict(X_val_fold)
    y_pred_val_classes = to_categorical(np.argmax(y_pred_val, axis=1), num_classes=2)

    # Calcular las métricas para el conjunto de validación
    accuracy_val = accuracy_score(y_val_fold, y_pred_val_classes) * 100
    f1_val = f1_score(y_val_fold, y_pred_val_classes, average='weighted') * 100
    precision_val = precision_score(y_val_fold, y_pred_val_classes, average='weighted') * 100
    recall_val = recall_score(y_val_fold, y_pred_val_classes, average='weighted') * 100

    # Almacenar las métricas de validación
    accuracies.append(accuracy_val)
    f1_scores.append(f1_val)
    precisions.append(precision_val)
    recalls.append(recall_val)

    # Hacer predicciones en el conjunto de prueba externo
    y_pred_test = model.predict(X_test_expanded)
    y_pred_test_classes = to_categorical(np.argmax(y_pred_test, axis=1), num_classes=2)

    # Calcular las métricas para el conjunto de prueba
    accuracy_test = accuracy_score(y_test_categorical, y_pred_test_classes) * 100
    f1_test = f1_score(y_test_categorical, y_pred_test_classes, average='weighted') * 100
    precision_test = precision_score(y_test_categorical, y_pred_test_classes, average='weighted') * 100
    recall_test = recall_score(y_test_categorical, y_pred_test_classes, average='weighted') * 100

    # Almacenar las métricas del conjunto de prueba
    test_accuracies.append(accuracy_test)
    test_f1_scores.append(f1_test)
    test_precisions.append(precision_test)
    test_recalls.append(recall_test)

# Función para formatear las métricas
def format_metric(mean, std):
    return f'{mean:.2f} ± {std:.2f}'

# Calcular las métricas medias y desviaciones estándar para la validación cruzada
accuracy_mean_std = format_metric(np.mean(accuracies), np.std(accuracies))
f1_mean_std = format_metric(np.mean(f1_scores), np.std(f1_scores))
precision_mean_std = format_metric(np.mean(precisions), np.std(precisions))
recall_mean_std = format_metric(np.mean(recalls), np.std(recalls))

# Calcular las métricas medias y desviaciones estándar para el conjunto de prueba
test_accuracy_mean_std = format_metric(np.mean(test_accuracies), np.std(test_accuracies))
test_f1_mean_std = format_metric(np.mean(test_f1_scores), np.std(test_f1_scores))
test_precision_mean_std = format_metric(np.mean(test_precisions), np.std(test_precisions))
test_recall_mean_std = format_metric(np.mean(test_recalls), np.std(test_recalls))

print(f'\nMétricas de Validación Cruzada:')
print(f'Precisión: {accuracy_mean_std}%')
print(f'Precisión (Precision): {precision_mean_std}%')
print(f'Recall: {recall_mean_std}%')
print(f'F1-score: {f1_mean_std}%')


# Imprimir las métricas del conjunto de prueba
print(f'\nMétricas del Conjunto de Prueba:')
print(f'Precisión: {test_accuracy_mean_std}%')
print(f'Precisión (Precision): {test_precision_mean_std}%')
print(f'Recall: {test_recall_mean_std}%')
print(f'F1-score: {test_f1_mean_std}%')


Fold 1
4/4 [==============================] - 0s 18ms/step
Fold 2
4/4 [==============================] - 0s 3ms/step
Fold 3
4/4 [==============================] - 0s 3ms/step
Fold 4
4/4 [==============================] - 0s 3ms/step
Fold 5
4/4 [==============================] - 0s 3ms/step

Métricas de Validación Cruzada:
Precisión: 89.11 ± 2.13%
Precisión (Precision): 89.58 ± 2.13%
Recall: 89.11 ± 2.13%
F1-score: 89.05 ± 2.21%

Métricas del Conjunto de Prueba:
Precisión: 83.92 ± 1.91%
Precisión (Precision): 83.95 ± 1.95%
Recall: 83.92 ± 1.91%
F1-score: 83.89 ± 1.95%


#Transformer

In [17]:

# OPTIMIZER
from tensorflow.keras import layers

#Pequeños :3

LEARNING_RATE_2 = 1e-3
WEIGHT_DECAY_2 = 1e-4
LAYER_NORM_EPS_2 = 1e-6


PROJECTION_DIM_2 = 256 # No se puede modificar

#Potencias de 2

NUM_HEADS_2 = 4
NUM_LAYERS_2 = 4






MLP_UNITS_2 = [
    PROJECTION_DIM_2 * 2,
    PROJECTION_DIM_2
]
def mlp(x, dropout_rate, hidden_units):
    # Iterate over the hidden units and
    # add Dense => Dropout.
    for units in hidden_units:
        x = layers.Dense(units, activation="selu")(x)
        x = layers.Dropout(dropout_rate)(x)
    return x
def transformer_2(encoded_patches):

    x1 = layers.LayerNormalization(epsilon=LAYER_NORM_EPS_2)(encoded_patches)

    # Multi Head Self Attention layer 1.
    attention_output = layers.MultiHeadAttention(
        num_heads=NUM_HEADS_2, key_dim=PROJECTION_DIM_2, dropout=0.1
    )(x1, x1)

    # Skip connection 1.
    x2 = layers.Add()([attention_output, encoded_patches])
    # Layer normalization 2.
    x3 = layers.LayerNormalization(epsilon=LAYER_NORM_EPS_2)(x2)

    # MLP layer 1.
    x4 = mlp(x3, hidden_units=MLP_UNITS_2, dropout_rate=0.1)
    # Skip connection 2.
    encoded_patches = layers.Add()([x4, x2])
    return encoded_patches

def Transform_sh_2(inputs):
      # Iterate over the number of layers and stack up blocks of
      # Transformer.
    for i in range(NUM_LAYERS_2):
          # Add a Transformer block.
        encoded_patches = transformer_2(inputs)

    return encoded_patches

In [18]:


# RED NEURONAL
def FullyConnected():

  inputs = Input(shape=(X_train.shape[1],1), name="input_1")


  Layer_1=tf.keras.layers.Conv1D(8,3,activation="selu",padding="same")(inputs)


#16

  Layer_1=tf.keras.layers.Conv1D(16,3,activation="selu",padding="same")(Layer_1)

#32

  Layer_1=tf.keras.layers.Conv1D(32,3,activation="selu",padding="same")(Layer_1)
  Pool_1=tf.keras.layers.MaxPool1D(2)(Layer_1)
  Pool_1=tf.keras.layers.Dropout(rate=0.5)(Pool_1)

#64

  Layer_1=tf.keras.layers.Conv1D(64,3,activation="selu",padding="same")(Pool_1)
  Pool_1=tf.keras.layers.MaxPool1D(2)(Layer_1)
  Pool_1=tf.keras.layers.Dropout(rate=0.5)(Pool_1)

#128

  Layer_1=tf.keras.layers.Conv1D(128,3,activation="selu",padding="same")(Pool_1)
  Pool_1=tf.keras.layers.MaxPool1D(2)(Layer_1)
  Pool_1=tf.keras.layers.Dropout(rate=0.5)(Pool_1)

#256

  Layer_1=tf.keras.layers.Conv1D(256,3,activation="selu",padding="same")(Pool_1)
  Pool_1=tf.keras.layers.MaxPool1D(2)(Layer_1)
  Pool_1=tf.keras.layers.Dropout(rate=0.5)(Pool_1)




  CVT1=Transform_sh_2(Pool_1)
  #CVT1=Transform_sh_2(CVT1)


  representation = tf.keras.layers.LayerNormalization(epsilon=LAYER_NORM_EPS_2)(CVT1)
  Flat=tf.keras.layers.GlobalMaxPooling1D()(representation)
  outputs = predictions = Dense(len(classes), activation="softmax", name="output_1")(Flat)
  model = Model(inputs = inputs, outputs = outputs)
  optimizer=Adam(learning_rate=0.001)
  model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
  #model.compile(optimizer=optimizer,loss=[loss_fn,'mse'],loss_weights = [1. ,0.0005],metrics=['accuracy'])
  return model

In [19]:
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn import decomposition
import numpy as np
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint

# Definición de clases
classes = [0, 1]

# Configuración de la validación cruzada y los datos
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Separación de características y etiquetas
Labels = df['Tipo de extraccióin global']
Features = df.drop(['Tipo de extraccióin global'], axis=1)

# División del dataset en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(Features, Labels, test_size=0.2, stratify=Labels, random_state=42)

scaler = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
# Aplicación de PCA
pca = decomposition.PCA(n_components=0.96, svd_solver='full', tol=1e-4)
pca.fit(X_train)
X_train = pca.transform(X_train)
X_test = pca.transform(X_test)

# Listas para almacenar las métricas de validación cruzada
accuracies = []
f1_scores = []
precisions = []
recalls = []

# Listas para almacenar las métricas del conjunto de prueba
test_accuracies = []
test_f1_scores = []
test_precisions = []
test_recalls = []

# Inicialización del fold
fold_no = 1

# Realizar la validación cruzada
for train_index, val_index in kf.split(X_train):
    print(f'Fold {fold_no}')
    fold_no += 1

    # División de los datos en entrenamiento y validación
    X_train_fold = X_train[train_index]
    y_train_fold = y_train.iloc[train_index]
    smote = SMOTE(random_state=42)
    X_train_fold, y_train_fold = smote.fit_resample(X_train_fold, y_train_fold)
    X_val_fold = X_train[val_index]
    y_val_fold = y_train.iloc[val_index]

    # Expansión de dimensiones para la red neuronal
    X_train_fold = np.expand_dims(X_train_fold, axis=-1)
    X_val_fold = np.expand_dims(X_val_fold, axis=-1)
    X_test_expanded = np.expand_dims(X_test, axis=-1)

    # Conversión de las etiquetas a categóricas
    y_train_fold = to_categorical(y_train_fold.astype(int), 2)
    y_val_fold = to_categorical(y_val_fold.astype(int), 2)
    y_test_categorical = to_categorical(y_test.astype(int), 2)

    # Inicialización del modelo
    model = FullyConnected()

    # Configuración del callback para guardar el mejor modelo basado en la métrica de validación
    checkpoint = ModelCheckpoint(
        'best_model.h5',  # Nombre del archivo donde se guarda el mejor modelo
        monitor='val_loss',  # Métrica a monitorear
        save_best_only=True,  # Guardar solo el mejor modelo
        mode='min',  # Modo de la métrica (minimizar la pérdida)
        verbose=0  # Mostrar mensajes de guardado
    )

    # Entrenamiento del modelo con validación y callback
    model.fit(
        X_train_fold,
        y_train_fold,
        epochs=500,
        batch_size=256,
        validation_data=(X_val_fold, y_val_fold),  # Validación en cada época
        callbacks=[checkpoint],  # Incluir el callback
        verbose=0
    )

    # Cargar el mejor modelo guardado
    model.load_weights('best_model.h5')

    # Hacer predicciones en el conjunto de validación
    y_pred_val = model.predict(X_val_fold)
    y_pred_val_classes = to_categorical(np.argmax(y_pred_val, axis=1), num_classes=2)

    # Calcular las métricas para el conjunto de validación
    accuracy_val = accuracy_score(y_val_fold, y_pred_val_classes) * 100
    f1_val = f1_score(y_val_fold, y_pred_val_classes, average='weighted') * 100
    precision_val = precision_score(y_val_fold, y_pred_val_classes, average='weighted') * 100
    recall_val = recall_score(y_val_fold, y_pred_val_classes, average='weighted') * 100

    # Almacenar las métricas de validación
    accuracies.append(accuracy_val)
    f1_scores.append(f1_val)
    precisions.append(precision_val)
    recalls.append(recall_val)

    # Hacer predicciones en el conjunto de prueba externo
    y_pred_test = model.predict(X_test_expanded)
    y_pred_test_classes = to_categorical(np.argmax(y_pred_test, axis=1), num_classes=2)

    # Calcular las métricas para el conjunto de prueba
    accuracy_test = accuracy_score(y_test_categorical, y_pred_test_classes) * 100
    f1_test = f1_score(y_test_categorical, y_pred_test_classes, average='weighted') * 100
    precision_test = precision_score(y_test_categorical, y_pred_test_classes, average='weighted') * 100
    recall_test = recall_score(y_test_categorical, y_pred_test_classes, average='weighted') * 100

    # Almacenar las métricas del conjunto de prueba
    test_accuracies.append(accuracy_test)
    test_f1_scores.append(f1_test)
    test_precisions.append(precision_test)
    test_recalls.append(recall_test)

# Función para formatear las métricas
def format_metric(mean, std):
    return f'{mean:.2f} ± {std:.2f}'

# Calcular las métricas medias y desviaciones estándar para la validación cruzada
accuracy_mean_std = format_metric(np.mean(accuracies), np.std(accuracies))
f1_mean_std = format_metric(np.mean(f1_scores), np.std(f1_scores))
precision_mean_std = format_metric(np.mean(precisions), np.std(precisions))
recall_mean_std = format_metric(np.mean(recalls), np.std(recalls))

# Calcular las métricas medias y desviaciones estándar para el conjunto de prueba
test_accuracy_mean_std = format_metric(np.mean(test_accuracies), np.std(test_accuracies))
test_f1_mean_std = format_metric(np.mean(test_f1_scores), np.std(test_f1_scores))
test_precision_mean_std = format_metric(np.mean(test_precisions), np.std(test_precisions))
test_recall_mean_std = format_metric(np.mean(test_recalls), np.std(test_recalls))

print(f'\nMétricas de Validación Cruzada:')
print(f'Precisión: {accuracy_mean_std}%')
print(f'Precisión (Precision): {precision_mean_std}%')
print(f'Recall: {recall_mean_std}%')
print(f'F1-score: {f1_mean_std}%')


# Imprimir las métricas del conjunto de prueba
print(f'\nMétricas del Conjunto de Prueba:')
print(f'Precisión: {test_accuracy_mean_std}%')
print(f'Precisión (Precision): {test_precision_mean_std}%')
print(f'Recall: {test_recall_mean_std}%')
print(f'F1-score: {test_f1_mean_std}%')


Fold 1
4/4 [==============================] - 0s 6ms/step
Fold 2
4/4 [==============================] - 0s 4ms/step
Fold 3
4/4 [==============================] - 0s 4ms/step
Fold 4
4/4 [==============================] - 0s 3ms/step
Fold 5
4/4 [==============================] - 0s 4ms/step

Métricas de Validación Cruzada:
Precisión: 88.85 ± 2.69%
Precisión (Precision): 88.90 ± 2.74%
Recall: 88.85 ± 2.69%
F1-score: 88.86 ± 2.70%

Métricas del Conjunto de Prueba:
Precisión: 84.12 ± 1.80%
Precisión (Precision): 84.27 ± 1.77%
Recall: 84.12 ± 1.80%
F1-score: 84.10 ± 1.83%


#Ml models

In [20]:
import numpy as np
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import MinMaxScaler, Normalizer
from imblearn.over_sampling import SMOTE  # Para el sobremuestreo de datos desequilibrados

# Carga de datos y preprocesamiento
Labels = df['Tipo de extraccióin global']
Features = df.drop(['Tipo de extraccióin global'],axis=1)
X_train, X_test, y_train, y_test = train_test_split(Features, Labels, test_size=0.2, stratify=Labels, random_state=42)
X = np.concatenate((X_train, X_test))
y = np.concatenate((y_train, y_test))

# Escalamiento de los datos
scaler = MinMaxScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# Lista de clasificadores
classifiers = [
    DecisionTreeClassifier(criterion='gini', max_depth=5, min_samples_leaf=1, min_samples_split=2),
    MLPClassifier(activation='relu', hidden_layer_sizes=(100, 50), learning_rate='adaptive', solver='adam'),
    KNeighborsClassifier(algorithm='auto', leaf_size=1, n_neighbors=3, p=2, weights='distance'),
    SGDClassifier(alpha=0.01, loss='hinge', max_iter=1000, penalty='l1'),
    ExtraTreesClassifier(max_depth=20, min_samples_split=4, n_estimators=250, random_state=40),
    RandomForestClassifier(max_depth=None, min_samples_split=2, n_estimators=150, random_state=50),
    GradientBoostingClassifier(learning_rate=0.1, max_depth=5, n_estimators=100, random_state=40)
]

# Definir los scorers
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score, average='weighted'),
    'recall': make_scorer(recall_score, average='weighted'),
    'f1': make_scorer(f1_score, average='weighted')
}

# Validación cruzada y cálculo de métricas
for clf in classifiers:
    # Validación cruzada en el conjunto de entrenamiento
    scores = cross_validate(clf, X_train, y_train, cv=5, scoring=scoring, return_train_score=False)

    print(f"Resultados de validación cruzada para {clf.__class__.__name__}:")
    for metric in scoring.keys():
        mean_score = np.mean(scores[f'test_{metric}']) * 100  # Convertir a porcentaje
        std_score = np.std(scores[f'test_{metric}']) * 100  # Convertir a porcentaje
        print(f"{metric.capitalize()} (validación): {mean_score:.2f} ± {std_score:.2f}")
    print("-" * 40)

    # Realizar múltiples divisiones de train/test para estimar la desviación estándar en testeo
    test_scores = {metric: [] for metric in scoring.keys()}
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for train_index, test_index in kf.split(X, y):
        X_train_kf, X_test_kf = X[train_index], X[test_index]
        y_train_kf, y_test_kf = y[train_index], y[test_index]

        # Aplicar SMOTE para el sobremuestreo en el conjunto de entrenamiento
        smote = SMOTE(random_state=42)
        X_train_kf, y_train_kf = smote.fit_resample(X_train_kf, y_train_kf)

        # Entrenar el modelo
        clf.fit(X_train_kf, y_train_kf)
        y_pred = clf.predict(X_test_kf)

        # Calcular las métricas en el conjunto de testeo
        for metric_name, scorer in scoring.items():
            score = scorer._score_func(y_test_kf, y_pred) * 100  # Convertir a porcentaje
            test_scores[metric_name].append(score)

    print(f"Resultados en el conjunto de testeo para {clf.__class__.__name__}:")
    for metric_name in scoring.keys():
        mean_test_score = np.mean(test_scores[metric_name])
        std_test_score = np.std(test_scores[metric_name])
        print(f"{metric_name.capitalize()} (testeo): {mean_test_score:.2f} ± {std_test_score:.2f}")
    print("=" * 40)


Resultados de validación cruzada para DecisionTreeClassifier:
Accuracy (validación): 84.45 ± 3.21
Precision (validación): 84.87 ± 3.15
Recall (validación): 84.45 ± 3.21
F1 (validación): 84.30 ± 3.36
----------------------------------------
Resultados en el conjunto de testeo para DecisionTreeClassifier:
Accuracy (testeo): 86.33 ± 2.61
Precision (testeo): 82.68 ± 3.26
Recall (testeo): 86.59 ± 3.32
F1 (testeo): 84.56 ± 2.99
Resultados de validación cruzada para MLPClassifier:
Accuracy (validación): 91.46 ± 1.90
Precision (validación): 91.59 ± 1.89
Recall (validación): 91.46 ± 1.90
F1 (validación): 91.44 ± 1.91
----------------------------------------
Resultados en el conjunto de testeo para MLPClassifier:
Accuracy (testeo): 89.22 ± 4.05
Precision (testeo): 88.90 ± 9.87
Recall (testeo): 88.07 ± 7.95
F1 (testeo): 87.71 ± 4.05
Resultados de validación cruzada para KNeighborsClassifier:
Accuracy (validación): 72.80 ± 3.89
Precision (validación): 72.85 ± 3.82
Recall (validación): 72.80 ± 3.89